# Advanced Production RAG Pipeline

This architecture natively handles complex document structures, multimodal extraction, and secure deterministic PII handling.

### Architecture Components:
1. **Docling:** Parses documents as Abstract Syntax Trees (ASTs), handling headers, tables, and extracting embedded images.
2. **Microsoft Presidio:** Analyzes text for Personally Identifiable Information (PII) and replaces it with surrogate tokens, routing the real data to an isolated Vault DB.
3. **All-Qdrant Vector Engine:** Handles both dense (Nomic) and sparse (FastEmbed/BM25) vectors locally.
4. **Early-Stage Re-constitution:** Hydrates the PII surrogate tokens in-memory right before passing the context to the local LLM (Gemma 4).
5. **BAML & Langfuse:** Guarantees structured JSON outputs and provides full telemetry.

In [ ]:
!pip install -qU docling presidio-analyzer presidio-anonymizer qdrant-client fastembed sentence-transformers langchain-openai baml-py langfuse

### 1. PII Detection & Secure Vaulting (Presidio)

In [ ]:
import hashlib
from presidio_analyzer import AnalyzerEngine

# Initialize Presidio Analyzer
analyzer = AnalyzerEngine()

# Simulated Secure PII Vault (In production, this is an encrypted Postgres DB)
PII_VAULT = {}

def anonymize_and_vault(text: str) -> str:
    """
    Detects PII, stores the real value in the secure vault, 
    and replaces the text with a deterministic surrogate token.
    """
    # Analyze text for entities like PERSON, EMAIL, PHONE_NUMBER
    results = analyzer.analyze(text=text, language='en')
    
    # Sort in reverse to avoid index shifting during string replacement
    results = sorted(results, key=lambda x: x.start, reverse=True)
    
    masked_text = text
    for res in results:
        real_value = text[res.start:res.end]
        entity_type = res.entity_type
        
        # Create a deterministic ID (e.g. <PERSON_a1b2c3>)
        # A salt should be added in production to prevent brute-forcing
        token_hash = hashlib.sha256(real_value.encode()).hexdigest()[:8]
        surrogate_token = f"<{entity_type}_{token_hash}>"
        
        # Store in Vault DB
        PII_VAULT[surrogate_token] = real_value
        
        # Inject surrogate into the chunk
        masked_text = masked_text[:res.start] + surrogate_token + masked_text[res.end:]
        
    return masked_text

def reconstitute_pii(text: str) -> str:
    """
    Early-stage hydration: Looks up surrogate tokens in the Vault 
    and replaces them with real data in-memory.
    """
    unmasked_text = text
    for token_id, real_value in PII_VAULT.items():
        unmasked_text = unmasked_text.replace(token_id, real_value)
    return unmasked_text

### 2. Document Layout Detection & Chunking (Docling)
Docling preserves document hierarchy and isolates images so we can pass them to a Vision model for captioning.

In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.chunking import HybridChunker

# Configure Docling to extract images bounding boxes during layout analysis
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True

converter = DocumentConverter()

# 1. Parse a sample document (replace with your local PDF path)
# For demo purposes, we will mock the parsed hierarchical chunks below.
# result = converter.convert("sample_contract.pdf")

# 2. Use Docling's HybridChunker to split text natively by Document Layout Headers
chunker = HybridChunker()
# chunks = chunker.chunk(result.document)

print("Docling initialized. Ready for hierarchical parsing.")

### 3. Pipeline Ingestion (Mock Data)
We simulate extracting a hierarchical chunk that contains sensitive PII and an embedded image tag.

In [ ]:
raw_docling_chunks = [
    "## Contract Terms\nThe primary point of contact is John Smith. His phone number is 555-0199 and email is jsmith@example.com.",
    "## Financial Summary\n[Embedded Figure: Bar Chart. VLM Extraction: Q3 Revenue grew by 15% to $2.1M driven by software sales.]"
]

anonymized_chunks = []
for chunk in raw_docling_chunks:
    # Pass the layout-aware chunk through Presidio before it touches embeddings
    safe_chunk = anonymize_and_vault(chunk)
    anonymized_chunks.append(safe_chunk)
    
print("--- Original Text ---")
print(raw_docling_chunks[0])
print("\n--- Masked Text (Ready for Qdrant) ---")
print(anonymized_chunks[0])
print("\n--- PII Vault Contents ---")
print(PII_VAULT)

### 4. Qdrant Fusion & LLM Orchestration

In [ ]:
from fastembed import SparseTextEmbedding
from sentence_transformers import CrossEncoder
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient, models

# Initialize Local Models
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
dense_embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1", 
    api_key="lm-studio", 
    model="text-embedding-nomic"
)

# Connect to Qdrant
client = QdrantClient(":memory:") # Using in-memory for the notebook demo
COLLECTION_NAME = "secure_documents"

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": models.VectorParams(size=768, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)}
)

# Embed and Index the ANONYMIZED chunks
dense_vectors = dense_embeddings.embed_documents(anonymized_chunks)
sparse_vectors = list(sparse_model.embed(anonymized_chunks))

points = []
for i, safe_chunk in enumerate(anonymized_chunks):
    points.append(models.PointStruct(
        id=i + 1,
        vector={
            "dense": dense_vectors[i],
            "sparse": models.SparseVector(indices=sparse_vectors[i].indices.tolist(), values=sparse_vectors[i].values.tolist())
        },
        payload={"chunk_text": safe_chunk} # Storing masked text in DB
    ))
client.upsert(collection_name=COLLECTION_NAME, points=points)
print("Secure indexing complete.")

In [ ]:
import os
from langfuse.decorators import observe

# Provide dummy keys to allow local script execution if not using Langfuse cloud
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-dummy"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-dummy"

@observe(as_type="retrieval")
def secure_retrieve_and_rerank(query: str):
    # 1. Embed query
    dense_query = dense_embeddings.embed_query(query)
    sparse_query_obj = list(sparse_model.embed([query]))[0]
    sparse_query = models.SparseVector(indices=sparse_query_obj.indices.tolist(), values=sparse_query_obj.values.tolist())

    # 2. Qdrant RRF Fusion
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(query=dense_query, using="dense", limit=5),
            models.Prefetch(query=sparse_query, using="sparse", limit=5),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=3,
        with_payload=True
    )
    
    masked_docs = [hit.payload["chunk_text"] for hit in results.points]
    if not masked_docs:
        return []

    # 3. Cross-Encoder Reranking (using masked docs is fine since semantics are preserved)
    rerank_pairs = [[query, doc] for doc in masked_docs]
    ce_scores = cross_encoder.predict(rerank_pairs)
    scored_docs = list(zip(masked_docs, ce_scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    
    # 4. Early-Stage Re-constitution: Unmask the winning chunks before giving them to the LLM
    top_chunk = scored_docs[0][0]
    hydrated_chunk = reconstitute_pii(top_chunk)
    
    return hydrated_chunk

# Note: In a complete project, you would call your BAML client here 
# using the `hydrated_chunk` as the context.

test_query = "Who is the primary contact and what is their phone number?"
final_context_for_llm = secure_retrieve_and_rerank(test_query)

print("\n--- Query ---")
print(test_query)
print("\n--- Hydrated Context Supplied to Gemma 4 (BAML) ---")
print(final_context_for_llm)